In [1]:
import json
import time
import os
from pathlib import Path
from requests.exceptions import HTTPError
from web3 import Web3
from concurrent.futures import ThreadPoolExecutor, as_completed
from hexbytes import HexBytes
from web3.datastructures import AttributeDict
from eth_abi import decode
from dotenv import load_dotenv

load_dotenv()  # Loads variables from .env into environment

INFURA_API_KEY = os.getenv("INFURA_API_KEY")
if not INFURA_API_KEY:
    raise Exception("INFURA_API_KEY not found in environment variables.")

# === CONFIGURATION ===
### URL
INFURA_URL = f"https://mainnet.infura.io/v3/{INFURA_API_KEY}"
CONTRACT_ADDRESS = POOL_ADDRESS = "0xCBCdF9626bC03E24f779434178A73a0B4bad62eD"
### FOLDER
OUT_FOLDER = 'out'
### FILES
ABI_FILE = f"{OUT_FOLDER}/WETH_WBTC_pool.json"  # Load your contract ABI file
ABI_FILE = f"{OUT_FOLDER}/uniswapFactory.json"  # Load your contract ABI file
BLOCKS_FILE = f"{OUT_FOLDER}/processed_blocks.json"
TRANSACTIONS_FILE = f"{OUT_FOLDER}/transactions.json"
PROCESSED_TX_FILE = "processed_transaction_block.txt"
MINT_CALLS_FILE = "mint_calls.json"
### CONSTANTS
# Number of transactions to process before writing to disk
BATCH_SIZE = 1000  
# Define how many blocks to fetch in one run
BATCH_BLOCK = 1
# The contract was created on this block + 1 (12369821)
UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER = 12369820

# === CONNECT TO ETHEREUM NODE ===
w3 = Web3(Web3.HTTPProvider(INFURA_URL))

assert w3.is_connected(), "Web3 provider connection failed"


# === UTILITY FUNCTIONS ===
# LOAD CONTRACT ABI
def load_contract_abi(file_path):
    """Load the contract ABI from a JSON file."""
    try:
        with open(file_path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise Exception(f"⚠️ ABI file '{file_path}' not found!")
    except json.JSONDecodeError:
        raise Exception(f"❌ Error parsing ABI JSON from '{file_path}'!")


contract_abi = load_contract_abi(ABI_FILE)
contract = w3.eth.contract(address=CONTRACT_ADDRESS, abi=contract_abi)

In [2]:
class HexBytesEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, HexBytes):
            return obj.hex()
        if isinstance(obj, AttributeDict):
            return dict(obj)  # Convert AttributeDict to a regular dictionary
        return super().default(obj)


def load_blocks_data(file_path):
    """Load block data from a JSON file as a dictionary mapping block numbers to block data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to integers (JSON keys are strings)
            return {int(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_blocks_data(blocks_data, file_path):
    """Save the blocks_data dictionary to a JSON file."""
    # Convert keys back to strings for JSON serialization.
    data_to_save = {str(k): v for k, v in blocks_data.items()}
    with open(file_path, "w") as f:
        json.dump(data_to_save, f, indent=2)


def fetch_block(block_number):
    """Fetch a single block with full transaction data using retries."""
    retries = 3
    while retries:
        try:
            block = w3.eth.get_block(block_number, full_transactions=True)
            print(f"Block {block_number} fetched with hash {block.hash.hex()}")
            # Process transactions: convert each transaction hash to a string.
            tx_hashes = []
            for tx in block.transactions:
                try:
                    # If tx is a dict with a 'hash' key
                    tx_hashes.append(tx["hash"].hex())
                except (TypeError, KeyError):
                    tx_hashes.append(tx.hex())
            return {
                "number": block.number,
                "hash": block.hash.hex(),
                "transactions": tx_hashes,
            }
        except HTTPError as e:
            if e.response.status_code == 429:
                print(f"Rate limit hit, retrying block {block_number} in 2 seconds...")
                time.sleep(2)
                retries -= 1
            else:
                print(f"Failed to fetch block {block_number}: {e}")
                raise
    print(f"Failed to fetch block {block_number} after retries")
    return None


def fetch_blocks(blocks_file, batch_block):
    """
    Fetch new blocks in batches and ensure data integrity.

    The block data is stored in a single file (blocks_file) as a dictionary mapping:
        block_number -> block_data

    The function first loads existing block data. It then checks for missing blocks in the
    range from the first block (which should equal UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER) to the
    highest block number already fetched. Any missing blocks are re-fetched.

    Then, new blocks are fetched starting from (max(existing_keys) + 1) up to batch_block count,
    or until the current chain's latest block.

    New blocks are written (streamed) every STREAM_BATCH blocks.
    """
    STREAM_BATCH = 200  # flush every 200 new blocks

    # Load existing block data as a dictionary {block_number: block_data}
    blocks_data = load_blocks_data(blocks_file)

    # Determine the starting block for re-fetching missing blocks:
    if blocks_data:
        first_block = min(blocks_data.keys())
        last_block = max(blocks_data.keys())
    else:
        # If no block has been fetched yet, start at the contract's creation block.
        first_block = UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER
        last_block = first_block - 1

    # Integrity check: find missing blocks in the range [first_block, last_block]
    expected_range = set(range(first_block, last_block + 1))
    fetched_blocks = set(blocks_data.keys())
    missing_blocks = sorted(expected_range - fetched_blocks)
    if missing_blocks:
        print(
            f"Integrity check: Missing {len(missing_blocks)} blocks in range {first_block}-{last_block}: {missing_blocks}"
        )
        # Re-fetch missing blocks in parallel.
        from concurrent.futures import ThreadPoolExecutor, as_completed

        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_block = {
                executor.submit(fetch_block, b): b for b in missing_blocks
            }
            for future in as_completed(future_to_block):
                b = future_to_block[future]
                result = future.result()
                if result:
                    blocks_data[b] = result
                    print(f"Recovered missing block {b}")
        # Save after re-fetching missing blocks.
        save_blocks_data(blocks_data, blocks_file)
        print("Integrity recovery completed: missing blocks have been re-fetched.")
    else:
        print("Integrity check passed: no missing blocks in the current range.")

    # Now, determine the starting block for new fetches:
    if blocks_data:
        new_start = max(blocks_data.keys()) + 1
    else:
        new_start = UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER

    latest_chain_block = w3.eth.block_number
    if new_start > latest_chain_block:
        print("No new blocks to fetch.")
        return blocks_data

    new_end = min(new_start + batch_block - 1, latest_chain_block)
    new_blocks_range = list(range(new_start, new_end + 1))
    print(
        f"Fetching new blocks from {new_start} to {new_end} (expected {len(new_blocks_range)} blocks)"
    )

    new_blocks_fetched = 0
    new_batch = {}
    from concurrent.futures import ThreadPoolExecutor, as_completed

    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_block = {
            executor.submit(fetch_block, b): b for b in new_blocks_range
        }
        for future in as_completed(future_to_block):
            b = future_to_block[future]
            result = future.result()
            if result:
                new_batch[b] = result
                new_blocks_fetched += 1
            if new_blocks_fetched % STREAM_BATCH == 0 and new_blocks_fetched:
                blocks_data.update(new_batch)
                save_blocks_data(blocks_data, blocks_file)
                print(f"Flushed {len(new_batch)} new blocks to {blocks_file}.")
                new_batch = {}
    # Merge any remaining new blocks.
    blocks_data.update(new_batch)
    save_blocks_data(blocks_data, blocks_file)
    print(f"Total new blocks fetched in this batch: {new_blocks_fetched}.")
    print("Fetching process completed.")
    return blocks_data

In [ ]:
# Example usage (in a Jupyter cell):
# Make sure UNISWAP_CONTRACT_BLOCK_CREATION_NUMBER is defined (e.g., 12369820)
updated_blocks = fetch_blocks(BLOCKS_FILE, BATCH_BLOCK)

In [3]:
def load_transactions_data(file_path):
    """Load transactions data from a JSON file as a dictionary mapping tx hash to transaction data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to strings (they should already be strings)
            return {str(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def load_blocks_data(file_path):
    """Load block data from a JSON file as a dictionary mapping block numbers to block data."""
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
            # Convert keys to integers (JSON keys are strings)
            return {int(k): v for k, v in data.items()}
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_transactions_data(tx_data, file_path):
    """Save the transactions data dictionary to a JSON file."""
    with open(file_path, "w") as f:
        json.dump(tx_data, f, indent=2, cls=HexBytesEncoder)


def fetch_transaction(tx_hash):
    """Fetch a single transaction using retries."""
    retries = 3
    while retries:
        try:
            tx = w3.eth.get_transaction(tx_hash)
            # Convert AttributeDict to dict if needed.
            return dict(tx)
        except HTTPError as e:
            if e.response.status_code == 429:
                time.sleep(2)
                retries -= 1
            else:
                print(f"Failed to fetch transaction {tx_hash}: {e}")
                raise
    return None


def process_block_transactions(block,tx_data, max_attempts=3):
    """
    For a given block (dict with a "transactions" list), fetch missing transactions.

    Retries up to max_attempts for any transactions that are still missing after a parallel fetch.
    Updates and returns the tx_data dictionary (mapping tx_hash -> tx data).
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    missing_txs = [tx for tx in block.get("transactions", []) if tx not in tx_data]
    attempt = 0
    while missing_txs and attempt < max_attempts:
        print(
            f"Block {block['number']}: Attempt {attempt+1} to fetch {len(missing_txs)} missing transactions."
        )
        new_tx_for_block = {}
        with ThreadPoolExecutor(max_workers=10) as executor:
            future_to_tx = {
                executor.submit(fetch_transaction, tx): tx for tx in missing_txs
            }
            for future in as_completed(future_to_tx):
                tx_hash = future_to_tx[future]
                result = future.result()
                if result:
                    new_tx_for_block[tx_hash] = result
        # Update our global transaction data with any newly fetched transactions.
        tx_data.update(new_tx_for_block)
        # Check again which transactions remain missing.
        missing_txs = [tx for tx in block.get("transactions", []) if tx not in tx_data]
        if missing_txs:
            print(
                f"Block {block['number']}: After attempt {attempt+1}, still missing {len(missing_txs)} transactions."
            )
        attempt += 1
    return tx_data


def process_transactions_per_block(blocks_file, transactions_file):
    """
    Process transactions block-by-block.

    For each block stored in blocks_file (a JSON dictionary mapping block number -> block data),
    the function checks the transactions list, fetches any missing transactions (with retries),
    and flushes the updated transactions data to transactions_file immediately after each block.

    This ensures that even if the process is interrupted, all transactions from processed blocks are saved.
    """
    # Load block data (as dictionary mapping block_number -> block_data)
    blocks_data = load_blocks_data(blocks_file)
    if not blocks_data:
        print("No blocks data available.")
        return {}

    # Load already processed transactions.
    tx_data = load_transactions_data(transactions_file)

    # Process blocks in sorted order.
    sorted_blocks = sorted(blocks_data.keys())
    total_new = 0

    for block_number in sorted_blocks:
        block = blocks_data[block_number]
        block_tx_hashes = block.get("transactions", [])
        # Determine which transactions in this block haven't been processed.
        missing_txs = [tx for tx in block_tx_hashes if tx not in tx_data]
        if not missing_txs:
            print(f"Block {block_number} already fully processed.")
        else:
            print(
                f"Processing block {block_number} with {len(missing_txs)} missing transactions."
            )
            # Process missing transactions with integrity check after processing the block.
            tx_data = process_block_transactions(block, tx_data)
            new_in_block = len([tx for tx in block_tx_hashes if tx in tx_data])
            print(f"Block {block_number}: Processed {new_in_block} transactions.")
            total_new += new_in_block

        # Flush the updated transaction data to disk after each block.
        save_transactions_data(tx_data, transactions_file)
        print(f"Flushed transactions from block {block_number} to {transactions_file}.")

    print(f"Total new transactions processed in this run: {total_new}.")
    return tx_data

In [4]:
# Example usage (in a Jupyter Notebook cell):
# Make sure your blocks_file and transactions_file are set up as expected.
processed_tx = process_transactions_per_block(BLOCKS_FILE, TRANSACTIONS_FILE)

Block 12369820 already fully processed.
Flushed transactions from block 12369820 to out/transactions.json.
Block 12369821 already fully processed.
Flushed transactions from block 12369821 to out/transactions.json.
Block 12369822 already fully processed.
Flushed transactions from block 12369822 to out/transactions.json.
Block 12369823 already fully processed.
Flushed transactions from block 12369823 to out/transactions.json.
Block 12369824 already fully processed.
Flushed transactions from block 12369824 to out/transactions.json.
Block 12369825 already fully processed.
Flushed transactions from block 12369825 to out/transactions.json.
Block 12369826 already fully processed.
Flushed transactions from block 12369826 to out/transactions.json.
Block 12369827 already fully processed.
Flushed transactions from block 12369827 to out/transactions.json.
Block 12369828 already fully processed.
Flushed transactions from block 12369828 to out/transactions.json.
Block 12369829 already fully processe

HTTPError: 402 Client Error: Payment Required for url: https://mainnet.infura.io/v3/3921fc62a7ce4cda98926f47409b3d19

In [ ]:
# Mint function signature and selector
target_types = "address,int24,int24,uint128,bytes"
function_name = "mint"

target_types = "uint16"
function_name = "increaseObservationCardinalityNext"

target_types = ('bytes[]',)
function_name = "multicall"

target_selector = Web3.keccak(text="multicall(bytes[])")[:4].hex()
print(target_selector)
# MINT_SELECTOR = "ac9650d8"

# ----- Helper Functions -----
def load_processed_tx_hashes(file_path):
    """Load processed transaction hashes from a text file (one per line)."""
    try:
        with open(file_path, "r") as f:
            return set(line.strip() for line in f if line.strip())
    except FileNotFoundError:
        return set()


def append_processed_tx_hashes(new_hashes, file_path):
    """Append a set of transaction hashes to a file, one per line."""
    with open(file_path, "a") as f:
        for tx in new_hashes:
            #f.write(tx + "\n")
            pass


def append_mint_calls(mint_calls, file_path):
    """Append mint call results to the output file, one JSON object per line."""
    with open(file_path, "a") as f:
        for call in mint_calls:
            f.write(json.dumps(call) + "\n")


def load_transactions_data(file_path):
    """Load the transactions data from a JSON file (assumed to be a dict of tx_hash -> tx data)."""
    try:
        with open(file_path, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def sanitize_value(value):
    """
    Recursively convert bytes to hex strings.
    """
    if isinstance(value, bytes):
        return value.hex()
    elif isinstance(value, (list, tuple)):
        return type(value)(sanitize_value(v) for v in value)
    else:
        return value


def decode_tx_by_function(tx, contract, target_function_name):
    """
    Decode a transaction's input using the contract ABI, returning a human-readable
    mapping of parameter names to values if the transaction calls the target function.

    Parameters:
      tx (dict): The transaction dictionary.
      contract: The contract instance (with ABI loaded).
      target_function_name (str): The name of the function to decode (e.g. "mint").

    Returns:
      dict or None: If the transaction calls the target function, returns a dict:
         {
           "transaction_hash": <tx hash>,
           "blockNumber": <block number>,
           "<target_function_name>_args": { <param1>: <value1>, <param2>: <value2>, ... }
         }
         Otherwise, returns None.
    """
    input_data = tx.get("input", "")
    if not input_data:
        return None
    try:
        # This will automatically try to decode the function input based on the contract ABI.
        func_obj, params = contract.decode_function_input(input_data)
        if func_obj.fn_name != target_function_name:
            return None
        # Build a mapping from parameter names to sanitized values.
        param_mapping = {}
        for inp in func_obj.abi.get("inputs", []):
            name = inp.get("name")
            value = params.get(name)
            param_mapping[name] = sanitize_value(value)
        return {
            "transaction_hash": tx.get("hash"),
            "blockNumber": tx.get("blockNumber"),
            f"{target_function_name}_args": param_mapping,
        }
    except Exception as e:
        print(f"Error decoding {target_function_name} call in tx {tx.get('hash')}: {e}")
        return None


def process_single_transaction(tx, target_selector, target_types, function_name):
    """
    Process a single transaction.
    If the transaction's input field starts with the target selector, decode the parameters
    according to target_types and return a dict with the function call details;
    otherwise return None.

    Parameters:
      tx (dict): A transaction dictionary.
      target_selector (str): The 4-byte function selector (as hex string, e.g. "0xabcdef12").
      target_types (list): A list of ABI types for the function's parameters
                           (e.g. ["address", "int24", "int24", "uint128", "bytes"]).
      function_name (str): A label for the function being decoded (e.g. "mint").

    Returns:
      dict or None: A dictionary containing the transaction hash, block number, and a key
                    named "<function_name>_args" mapped to the decoded parameters, or None if not matching.
    """
    input_data = tx.get("input", "")
    if input_data and input_data.startswith(target_selector):
        # Remove the "0x" and the selector (first 10 characters: "0x" + 8 hex digits)
        data_without_selector = input_data[8:]
        try:
            # Decode the parameters using the provided types.
            raw_params = decode(target_types, bytes.fromhex(data_without_selector))
            # Sanitize: convert any bytes into hex strings.
            sanitized_params = tuple(sanitize_value(p) for p in raw_params)
            # Create a mapping from parameter names to values.
            param_mapping = {
                name: value for name, value in zip(parameter_names, sanitized_params)
            }

            return {
                "transaction_hash": tx.get("hash"),
                "blockNumber": tx.get("blockNumber"),
                f"{function_name}_args": sanitized_params,
            }
        except Exception as e:
            print(f"Error decoding {function_name} call in tx {tx.get('hash')}: {e}")
            return None
    return None


def process_transactions_in_batches(
    transactions_file,
    processed_tx_file,
    mint_calls_file,
    batch_size,
    target_selector,
    target_types,
    function_name,
):
    """
    Process transactions from transactions_file to find Mint calls.

    - Loads the transactions (as a dict mapping tx_hash -> tx data).
    - Loads already processed transaction hashes from processed_tx_file.
    - Iterates over transactions that haven't been processed.
    - In parallel, processes each transaction to see if it is a Mint call.
    - Every batch_size transactions processed, flush the results to mint_calls_file
      (one JSON object per line) and append the processed transaction hashes to processed_tx_file.
    """
    all_tx = load_transactions_data(transactions_file)
    processed = load_processed_tx_hashes(processed_tx_file)

    # Only process transactions that have not been processed yet.
    tx_list = [tx for tx in all_tx.values() if tx.get("hash") not in processed]
    print(f"Total transactions to process: {len(tx_list)}")

    processed_in_batch = set()
    mint_calls_batch = []
    total_processed = 0

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(process_single_transaction, tx, target_selector, target_types, function_name): tx
            for tx in tx_list
        }
        for future in as_completed(futures):
            tx = futures[future]
            tx_hash = tx.get("hash")
            processed_in_batch.add(tx_hash)
            result = future.result()
            if result is not None:
                mint_calls_batch.append(result)
            total_processed += 1

            # If we've processed a batch, flush the results.
            if total_processed % batch_size == 0:
                if mint_calls_batch:
                    print(type(mint_calls_batch))
                    print(mint_calls_batch)
                    append_mint_calls(mint_calls_batch, mint_calls_file)
                    print(
                        f"Flushed {len(mint_calls_batch)} mint call entries to {mint_calls_file}."
                    )
                    mint_calls_batch = []
                if processed_in_batch:
                    append_processed_tx_hashes(processed_in_batch, processed_tx_file)
                    processed_in_batch = set()

    # Flush any remaining entries after processing all transactions.
    if mint_calls_batch:
        append_mint_calls(mint_calls_batch, mint_calls_file)
        print(
            f"Flushed remaining {len(mint_calls_batch)} mint call entries to {mint_calls_file}."
        )
    if processed_in_batch:
        append_processed_tx_hashes(processed_in_batch, processed_tx_file)
        print(
            f"Updated processed transaction file with remaining {len(processed_in_batch)} entries."
        )

    print("Transaction processing complete.")

In [ ]:
# ----- File I/O Helper Functions -----
def load_transactions_data(file_path):
    """
    Load transactions data from a JSON file.
    The file is assumed to contain a JSON object mapping tx_hash → tx data.
    """
    try:
        with open(file_path, "r") as f:
            return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def group_transactions_by_block(tx_dict):
    """
    Group transactions by block number.
    Returns a dictionary mapping block number (int) to a list of transaction dictionaries.
    """
    groups = {}
    for tx in tx_dict.values():
        block = tx.get("blockNumber")
        if block is not None:
            groups.setdefault(block, []).append(tx)
    return groups


def load_processed_blocks(file_path):
    """
    Load processed block numbers from a text file (one per line).
    Returns a set of block numbers.
    """
    try:
        with open(file_path, "r") as f:
            return set(int(line.strip()) for line in f if line.strip())
    except FileNotFoundError:
        return set()


def append_processed_block(block_number, file_path):
    """Append a processed block number to the tracking file."""
    with open(file_path, "a") as f:
        f.write(str(block_number) + "\n")


def append_function_calls(block_entry, file_path):
    """
    Append a block's decoded function call results to the output file.
    Each block's results are written as one JSON object per line.
    """
    with open(file_path, "a") as f:
        f.write(json.dumps(block_entry) + "\n")


# ----- Sanitization and Decoding Helpers -----
def sanitize_value(value):
    """
    Recursively convert bytes objects to hex strings.
    This ensures all values are JSON-serializable and human-readable.
    """
    if isinstance(value, bytes):
        return "0x" + value.hex()  # Prepend 0x for clarity.
    elif isinstance(value, (list, tuple)):
        return type(value)(sanitize_value(v) for v in value)
    else:
        return value


def decode_tx_by_function(tx, contract):
    """
    Decode a transaction's input using the contract's ABI.

    If the transaction calls a known function in the ABI, returns a dictionary:
      {
         "transaction_hash": <tx hash>,
         "blockNumber": <block number>,
         "function_name": <name of the function>,
         "args": { <param1>: <value1>, <param2>: <value2>, ... }
      }
    If the transaction does not call any function (or if decoding fails because the selector doesn't match),
    returns None.

    Note: input_data must include the "0x" prefix. If it doesn't, it is added.
    """
    input_data = tx.get("input", "")
    if not input_data:
        return None  # No input means no contract call.
    # Ensure the input has "0x" prefix.
    if not input_data.startswith("0x"):
        input_data = "0x" + input_data
    try:
        func_obj, params = contract.decode_function_input(input_data)
        # Build a mapping from parameter names to sanitized values.
        param_mapping = {
            inp.get("name"): sanitize_value(params.get(inp.get("name")))
            for inp in func_obj.abi.get("inputs", [])
        }
        return {
            "transaction_hash": tx.get("hash"),
            "blockNumber": tx.get("blockNumber"),
            "function_name": func_obj.fn_name,
            "args": param_mapping,
        }
    except Exception as e:
        # If error indicates no matching selector, return None.
        if "matching selector" in str(e):
            return None
        else:
            print(f"Error decoding function call in tx {tx.get('hash')}: {e}")
            return None


# ----- Main Block Processing Function -----
def process_transactions_by_block(
    transactions_file, processed_blocks_file, output_file, contract
):
    """
    Process transactions grouped by block, decoding every function call found in each transaction.

    For every block (from transactions_file) not already processed (tracked in processed_blocks_file):
      - Process all its transactions in parallel.
      - For each transaction, decode its function call using contract.decode_function_input.
      - Each decoded transaction returns the function name and a mapping of parameter names to values.
      - Even if no transaction in a block calls a known function, an entry for that block (with an empty list)
        is appended to output_file.
      - Once a block is fully processed, its block number is appended to processed_blocks_file.

    Each block's output is written as one JSON object per line in output_file.
    """
    all_tx = load_transactions_data(transactions_file)
    blocks_group = group_transactions_by_block(all_tx)
    processed_blocks = load_processed_blocks(processed_blocks_file)

    for block_number in sorted(blocks_group.keys()):
        if block_number in processed_blocks:
            print(f"Block {block_number} already processed; skipping.")
            continue

        txs = blocks_group[block_number]
        print(f"Processing block {block_number} with {len(txs)} transactions.")
        block_function_calls = []

        with ThreadPoolExecutor(max_workers=10) as executor:
            futures = {
                executor.submit(decode_tx_by_function, tx, contract): tx for tx in txs
            }
            for future in as_completed(futures):
                result = future.result()
                if result is not None:
                    block_function_calls.append(result)

        block_entry = {
            "blockNumber": block_number,
            "function_calls": block_function_calls,  # List of decoded function call dicts.
        }
        append_function_calls(block_entry, output_file)
        append_processed_block(block_number, processed_blocks_file)

    print("Block-based transaction processing complete.")

In [ ]:
# ----- Example Usage in a Jupyter Notebook Cell -----
# For instance, to process all transactions and decode every function call using the contract's ABI:
# Assuming your initialization cell has already defined:
#   - w3, contract, TRANSACTIONS_FILE, processed_blocks_file (e.g., "processed_transaction_block.txt"),
#     and output_file (e.g., "contract_calls.json").
#
# Run the following cell:
process_transactions_by_block(
    TRANSACTIONS_FILE,
    "processed_transaction_block.txt",  # File that tracks processed block numbers.
    "contract_calls.json",  # Output file with one JSON object per block.
    contract,
)

In [ ]:
input_data = "ac9650d80000000000000000000000000000000000000000000000000000000000000020000000000000000000000000000000000000000000000000000000000000000300000000000000000000000000000000000000000000000000000000000000600000000000000000000000000000000000000000000000000000000000000160000000000000000000000000000000000000000000000000000000000000022000000000000000000000000000000000000000000000000000000000000000c4f3995c67000000000000000000000000bb2b8038a1640196fbe3e38816f3e67cba72d940000000000000000000000000000000000000000000000000000007984fb931bd000000000000000000000000000000000000000000000000000000006091b2d1000000000000000000000000000000000000000000000000000000000000001ceabd638ef3d501502f1b0dc983f952c999fb677259401219ef0a187355c184ea0eba5c2e45a2ee2bdd1b9d20a742cf208b1c8005940ee08a9885053de4ffac0900000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000008413ead5620000000000000000000000002260fac5e5542a773aa44fbcfedf7c193bc2c599000000000000000000000000c02aaa39b223fe8d0a0e5c4f27ead9083c756cc20000000000000000000000000000000000000000000000000000000000000bb80000000000000000000000000000000000061b1a1c878c3621dac27d63b440af0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001a4d44f2bf2000000000000000000000000bb2b8038a1640196fbe3e38816f3e67cba72d940000000000000000000000000000000000000000000000000000007984fb931bd00000000000000000000000000000000000000000000000000000000000000640000000000000000000000002260fac5e5542a773aa44fbcfedf7c193bc2c599000000000000000000000000c02aaa39b223fe8d0a0e5c4f27ead9083c756cc20000000000000000000000000000000000000000000000000000000000000bb8000000000000000000000000000000000000000000000000000000000003dd88000000000000000000000000000000000000000000000000000000000004099800000000000000000000000000000000000000000000000000000000016e9bba00000000000000000000000000000000000000000000000027b16fbadbe2997e0000000000000000000000004bd047ca72fa05f0b89ad08fe5ba5ccdc07dffbf000000000000000000000000000000000000000000000000000000006091b2d1000000000000000000000000000000000000000000000000000000000000000100000000000000000000000000000000000000000000000000000000"
func_obj, params = contract.decode_function_input(input_data)

In [ ]:
for i in contract.abi:
    for b in i['inputs']:
        print(b["name"])

In [ ]:
input_data = "0xa9059cbb0000000000000000000000003db36ee573fa40ac5469c56bede4e71886a5aac2000000000000000000000000000000000000000000000000000000001312d000"
contract.decode_function_input(input_data)

In [ ]:
a = load_blocks_data(BLOCKS_FILE)

In [ ]:
len(a)

In [ ]:
b = list(a.keys())

In [ ]:
len(b)

In [ ]:
b.sort()

In [ ]:
b[0]

In [ ]:
b[-1]

In [ ]:
b[-1] - b[0]